# Predicting EV Purchases: Stacking and Submission

Notebook 3 of 3 for the Kaggle competition Playground Series S6E9 (predict `Will_Buy_EV`, scored by ROC AUC). This notebook compares our three models, adds out-of-fold predictions that other competitors published, chooses the members of a logistic stacker by forward selection and writes the submission. The code is the code of the submitted version, moved out of one notebook into three.

It reads the competition files (CC BY 4.0), `own_predictions.npz` from notebook 2 and seven public sources of out-of-fold predictions, and writes `submission.csv` and `stack.json` (the chosen models, their weights and the cross-validated AUC).

## Setup

Import the libraries and set the seed. `SMOKE` stops the forward selection after one added model for a quick check run on Kaggle; it is off in this version.

In [ ]:
import glob
import hashlib
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from scipy.special import expit
from scipy.stats import norm, rankdata
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict

warnings.filterwarnings("ignore")

SMOKE = False
SEED = 42
print(f"smoke run: {SMOKE}")
print(f"scikit-learn {sklearn.__version__}, pandas {pd.__version__}, numpy {np.__version__}")

**Conclusion.** The smoke switch is off. The Kaggle image provides scikit-learn 1.6.1, pandas 2.3.3 and NumPy 2.1.3, so nothing has to be installed.

## Load the data and our predictions

Read the competition files and the predictions saved by notebook 2. The row ids saved with the predictions must match `train.csv` and `test.csv` exactly; otherwise the notebook stops, which guards against reading a smoke run or an outdated version of notebook 2.

In [ ]:
root = glob.glob("/kaggle/input/**/train.csv", recursive=True)[0].rsplit("/", 1)[0]
train = pd.read_csv(f"{root}/train.csv")
test = pd.read_csv(f"{root}/test.csv")
sample = pd.read_csv(f"{root}/sample_submission.csv")

TARGET = "Will_Buy_EV"
y = (train[TARGET] == "Yes").astype(int).to_numpy()
print(train.shape, test.shape)
print(train.isna().sum().sum(), test.isna().sum().sum())
print(f"positive rate: {y.mean():.4f}")

own_file = glob.glob("/kaggle/input/**/own_predictions.npz", recursive=True)
assert len(own_file) == 1, own_file
saved = np.load(own_file[0])
assert np.array_equal(saved["train_id"], train["id"].to_numpy()), "train ids differ from notebook 2"
assert np.array_equal(saved["test_id"], test["id"].to_numpy()), "test ids differ from notebook 2"
oof_lgb, test_lgb = saved["oof_lgb"], saved["test_lgb"]
oof_xgb, test_xgb = saved["oof_xgb"], saved["test_xgb"]
oof_lin, test_lin = saved["oof_lin"], saved["test_lin"]
print(f"own predictions: {len(oof_lgb)} train rows and {len(test_lgb)} test rows, ids match train.csv and test.csv")

**Conclusion.** The competition files load as in notebooks 1 and 2 (668,665 and 286,571 rows, purchase rate 17.46%), and the predictions from notebook 2 cover exactly these rows in the same order: both id checks passed.

## Our models side by side

Before adding anything else, check how similar our three models are (rank correlation of their out-of-fold predictions) and how well a simple stacker does with them alone. Every prediction is turned into a normal-score rank, and the stacker is a logistic regression whose score is always measured with 3-fold cross-validation on the out-of-fold predictions.

In [ ]:
def gauss_rank(x):
    return norm.ppf((rankdata(x) - 0.5) / len(x))


own = {"LightGBM": (oof_lgb, test_lgb), "XGBoost": (oof_xgb, test_xgb), "Logistic": (oof_lin, test_lin)}
own_corr = pd.DataFrame(np.corrcoef([rankdata(o) for o, _ in own.values()]), index=list(own), columns=list(own))
print(own_corr.round(4).to_string())

stack_cv = StratifiedKFold(3, shuffle=True, random_state=SEED)


def stack_auc(cols, candidates):
    x = np.column_stack([candidates[c][0] for c in cols])
    pred = cross_val_predict(LogisticRegression(max_iter=2000), x, y, cv=stack_cv, method="decision_function")
    return roc_auc_score(y, pred)


candidates = {name: (gauss_rank(o), gauss_rank(t)) for name, (o, t) in own.items()}
own_auc = stack_auc(list(own), candidates)
print(f"stack of our three models: {own_auc:.5f}")

**Conclusion.** LightGBM and XGBoost rank the rows almost identically (rank correlation 0.9993), and the logistic regression differs more (0.9949 with LightGBM, 0.9948 with XGBoost). Stacking the three gives a cross-validated AUC of 0.94619, only 0.00002 above LightGBM alone (0.94617). Different algorithms on the same features make the same mistakes; to go further, the stack needs models built on different feature pipelines.

## Public out-of-fold libraries

Several competitors published the out-of-fold (OOF) and test predictions of their own models. An OOF prediction for a row is made by a model that never saw that row's label, so these files can be stacked honestly with our models. They were built with feature pipelines that differ from ours (triple target encoding, digit features, income windows, quantised income ladders, a RealMLP neural network, a linear-tree LightGBM and generator-aware linear models), which is the kind of diversity that helps a stack.

Sources used, with thanks to their authors:

- [S6E9 Six Feature Views OOF Library](https://www.kaggle.com/datasets/megayak/s6e9-six-feature-views-oof-library) by megayak (views A to F and a RealMLP), CC0
- [S6E9 hybrid LightGBM OOF](https://www.kaggle.com/datasets/megayak/s6e9-hybrid-lgbm-oof) by megayak, CC0
- [S6E9 digit-leak OOF](https://www.kaggle.com/datasets/megayak/s6e9-digit-leak-oof) by megayak, CC0
- [S6E9 residual stack OOF](https://www.kaggle.com/datasets/legtarrr/s6e9-residual-stack-oof) by legtarrr
- The outputs of three notebooks by Paul Bryan Elefante: [Generator-Aware Ridge Logistic Regression](https://www.kaggle.com/code/heuljax/kps6e09-generator-aware-ridge-logistic-regression), [Logistic Regression Sample](https://www.kaggle.com/code/heuljax/kps6e09-logistic-regression-sample) and [XGB Sample](https://www.kaggle.com/code/heuljax/kps6e09-xgb-sample). They use 10 folds with inner 5-fold cross-fitting for every label-based feature.

All files are aligned to `train.csv` and `test.csv` by id or by row order, and every column is turned into a normal-score rank so that probabilities and ranks can be mixed. The cell prints the OOF AUC of every public model next to our LightGBM.

In [ ]:
def find_file(name):
    return glob.glob(f"/kaggle/input/**/{name}", recursive=True)[0]


def load_csv(name, column):
    return pd.read_csv(find_file(name)).sort_values("id")[column].to_numpy()


start = time.time()
six_views = ["A_lgbm_triple_te_digits_3seed", "B_xgb_on_A_features", "C_no_digits_windows_lift_sm2_30_300",
             "D_no_exact_key_ladder_windows", "E_ladder25_250_2500_lift_sm5_50_500", "F_exact_rate_as_init_score"]
public = {f"six_views_{v[0]}": (load_csv("oof_six_views.csv", v), load_csv("test_six_views.csv", v)) for v in six_views}
public["realmlp"] = (load_csv("oof_realmlp_g.csv", "G_realmlp_3seed"), load_csv("test_realmlp_g.csv", "G_realmlp_3seed"))
public["hybrid_lgbm"] = (np.load(find_file("oof_hybrid_3seed.npy")), np.load(find_file("test_hybrid_3seed.npy")))
for model in ["lgb", "cat"]:
    public[f"digit_{model}"] = (np.load(find_file(f"oof_{model}.npy")), np.load(find_file(f"test_{model}.npy")))
for model in ["v19", "yek", "jaz"]:
    public[f"residual_{model}"] = (load_csv(f"{model}_oof.csv", "pred"), load_csv(f"{model}_test.csv", "pred"))
for name, prefix in [("heuljax_ridge", "GENERATOR_AWARE_LOGREG_SAMPLE"), ("heuljax_logreg", "LOGREG_SAMPLE"), ("heuljax_xgb", "XGB_SAMPLE")]:
    oof_part = pd.read_parquet(find_file(f"{prefix}_OOF.parquet")).sort_values("id")
    test_part = pd.read_parquet(find_file(f"{prefix}_TEST.parquet")).sort_values("id")
    public[name] = (oof_part["oof_pred"].to_numpy(), test_part["test_pred"].to_numpy())

for name, (o, t) in public.items():
    assert len(o) == len(train) and len(t) == len(test)
    candidates[name] = (gauss_rank(o), gauss_rank(t))
public_scores = pd.Series({name: roc_auc_score(y, o) for name, (o, _) in public.items()}).sort_values(ascending=False)
print(public_scores.round(5).to_frame("OOF AUC").to_string())
print(f"our LightGBM OOF AUC: {roc_auc_score(y, oof_lgb):.5f}")
print(f"public models loaded: {len(public)}")
print(f"loading elapsed: {time.time() - start:.0f} seconds")

**Conclusion.** All 16 public models loaded and line up with the competition rows. Their out-of-fold AUCs run from 0.94560 (`digit_cat`) to 0.94640 (the generator-aware ridge), and nine of them score above our LightGBM (0.94617). None is far above the rest, which suggests that none of them leaks the target.

## Stacking with forward selection

Start from our three models. At each step, try adding every remaining model to the stack and keep the one that raises the cross-validated AUC the most. Stop when no model adds more than 0.000002. The stacker is a logistic regression, and its score is always measured with 3-fold cross-validation on the out-of-fold predictions, so a model is only kept if it helps on rows the stacker did not see. The same cross-validation also chooses the members, so the final score is slightly optimistic; the leaderboard is the out-of-sample check.

In [ ]:
start = time.time()
chosen = list(own)
best_auc = own_auc
history = [("our three models", best_auc)]
while True:
    if SMOKE and len(chosen) > len(own):
        break
    trials = {name: stack_auc(chosen + [name], candidates) for name in candidates if name not in chosen}
    if not trials:
        break
    name, score = max(trials.items(), key=lambda item: item[1])
    if score <= best_auc + 2e-6:
        break
    chosen.append(name)
    best_auc = score
    history.append((f"+ {name}", score))
    print(f"add {name}: {score:.6f}")

print(pd.DataFrame(history, columns=["step", "stacked AUC"]).round(6).to_string())

meta = LogisticRegression(max_iter=2000)
meta.fit(np.column_stack([candidates[c][0] for c in chosen]), y)
weights = {name: round(float(w), 3) for name, w in zip(chosen, meta.coef_[0])}
test_stack = np.column_stack([candidates[c][1] for c in chosen])
print(weights)
print(f"forward selection elapsed: {time.time() - start:.0f} seconds")

**Conclusion.** Forward selection kept five public models and raised the cross-validated AUC from 0.946187 (our three models) to 0.946649. The generator-aware ridge gave the largest step (+0.000367), followed by the residual stack v19 (+0.000055), the heuljax XGBoost (+0.000028), the heuljax logistic regression (+0.000007) and view B of the six-view library (+0.000005). The ridge also gets the largest weight (1.232), while our logistic regression gets a negative weight (-0.289) once the stronger linear models are in. The residual stack v19 was chosen second although it is among the weaker models on its own (0.94596): a model built on a different view helps a stack even when it is weaker by itself. Because the same cross-validation also chose the members, 0.946649 is slightly optimistic; the leaderboard is the out-of-sample check.

## Submission

Apply the stacker to the test predictions and save the probabilities in the format of the sample submission. The chosen models, their weights and both cross-validated scores also go to `stack.json`.

In [ ]:
submission = pd.DataFrame({"id": test["id"], TARGET: expit(meta.decision_function(test_stack))})
assert submission.shape == sample.shape and submission[TARGET].between(0, 1).all()
submission.to_csv("submission.csv", index=False)
Path("stack.json").write_text(json.dumps({"chosen": chosen, "weights": weights, "own_stack_auc": own_auc,
                                          "stack_auc": best_auc}, indent=2), encoding="utf-8")
print(f"submission rows: {len(submission)}, mean predicted probability: {submission[TARGET].mean():.5f}")
print(f"submission.csv sha256 {hashlib.sha256(Path('submission.csv').read_bytes()).hexdigest()[:16]}")
submission.head()

**Conclusion.** The submission has 286,571 rows, one probability per test id, with a mean of 0.17459, close to the 17.46% purchase rate of the train set. It is the v6 stack: our LightGBM, XGBoost and logistic regression with the five selected public models, at a cross-validated AUC of 0.946649.

## Summary

Our three models stack to a cross-validated AUC of 0.94619, and five public out-of-fold models built on other feature pipelines lift it to 0.946649. The largest single step comes from the generator-aware ridge (+0.000367): a strong linear model sees this data differently from the tree models.

Some things did not help during the competition. An MLP in the stack (v3) and seed bagging (v4) left the leaderboard scores unchanged, and CatBoost cost about an hour per run for a small stack weight, so v5 and v6 dropped all three. The data is close to a noisy rule, so the room above this stack is small.